# ECE326 Cumulative Problem Set - Solutions

Reference solutions, explanations, and executable tests for Python foundations through Array Programming I. No Array Programming II prerequisites.


## Setup


In [ ]:
import numpy as np


## Q1 - Encode consecutive character runs


In [ ]:
def encode_runs(text):
    runs = []
    for character in text:
        if runs and runs[-1][0] == character:
            previous, count = runs[-1]
            runs[-1] = (previous, count + 1)
        else:
            runs.append((character, 1))
    return runs

def decode_runs(runs):
    return ''.join(character * count for character, count in runs)


The first character starts a run because the result list is empty. Later characters extend only the most recent run or begin a new one. A frequency dictionary loses run order and separated occurrences. Tuples are replaced, not mutated; decoding only reads the list. Encoding takes O(L) work and O(R) output storage for length L and R runs. Decoding is proportional to reconstructed length. A list comprehension followed by join is equally acceptable.


## Q2 - Reshape, sample, and flip


In [ ]:
def sample_frame(raw, h, w):
    frame = raw.reshape(h, w)
    selected = frame[::2, ::2]
    return selected[:, ::-1].copy()


Shapes are (h,w), then ((h+1)/*2,(w+1)*/2), then unchanged. Slices share storage, so the explicit final copy matters. Reversing before sampling differs for even widths. Work and output storage are O(h*w).


## Q3 - How many ways up the stairs?


In [ ]:
def count_steps(n, steps):
    cache = {0: 1}
    return _count_remaining(n, steps, cache)

def _count_remaining(remaining, steps, cache):
    if remaining < 0:
        return 0
    if remaining in cache:
        return cache[remaining]
    total = 0
    for step in steps:
        total += _count_remaining(remaining-step, steps, cache)
    cache[remaining] = total
    return total


Let F(r) count completions of remaining distance r. F(0)=1, F(r)=0 for r<0, and F(r) is the sum of F(r-s) over allowed s for positive r. Choosing the first step partitions the ordered sequences without double-counting. Each reachable nonnegative state is computed once and performs k helper calls; negative states terminate immediately. This gives O(n*k) unit-cost work for n>=1, at most n+1 cache entries, and O(n+1) recursion depth. Arbitrary-precision addition is not literally constant cost; this work bound uses a unit-cost convention. Cache membership must not be confused with truthiness: zero answers are cached too. A fresh cache avoids reusing results from different step tuples. Crucially, recursive calls go to the cached helper itself.


## Q4 - Find sustained overheating episodes


In [ ]:
def episodes_python(temperatures, threshold, min_length):
    result = []
    start = None
    for i, value in enumerate(temperatures):
        if value >= threshold:
            if start is None:
                start = i
        elif start is not None:
            if i - start >= min_length:
                result.append((start, i))
            start = None
    if start is not None and len(temperatures) - start >= min_length:
        result.append((start, len(temperatures)))
    return result

def episodes_numpy(temperatures, threshold, min_length):
    hot = temperatures >= threshold
    padded = np.zeros(len(hot) + 2, dtype=bool)
    padded[1:-1] = hot
    positions = np.arange(len(hot) + 1, dtype=np.int64)
    starts = positions[~padded[:-1] & padded[1:]]
    stops = positions[padded[:-1] & ~padded[1:]]
    keep = stops - starts >= min_length
    return starts[keep], stops[keep]


False sentinels make both endpoints ordinary transitions. A False-to-True transition starts an episode; True-to-False ends it at the exclusive stop. Starts and stops alternate and are paired in chronological order. Subtracting them gives lengths, and one shared mask filters both boundary arrays. The empty case yields no transitions. Both implementations take O(n) work; NumPy uses O(n) temporary storage while the Python traversal needs only O(1) extra state beyond output. Other boundary constructions are acceptable.


## Q5 - Find local peaks


In [ ]:
def local_peaks(heights):
    result = np.zeros(heights.shape, dtype=bool)
    if heights.shape[0] < 3 or heights.shape[1] < 3:
        return result
    center = heights[1:-1, 1:-1]
    peaks = ((center > heights[:-2, :-2])
             & (center > heights[:-2, 1:-1])
             & (center > heights[:-2, 2:])
             & (center > heights[1:-1, :-2])
             & (center > heights[1:-1, 2:])
             & (center > heights[2:, :-2])
             & (center > heights[2:, 1:-1])
             & (center > heights[2:, 2:]))
    result[1:-1, 1:-1] = peaks
    return result


All nine slices have shape (rows-2, columns-2). A mask entry is True only when its center exceeds all eight neighbors. Strict comparison excludes ties. Assign the interior mask to result[1:-1,1:-1] in an initially all-False Boolean array. Border cells are never candidates but can disqualify adjacent interior cells. The output does not share input storage. Work and temporary storage are O(rows*columns).


## Executable verification
These tests run in the generated notebook. Tests deliberately use loops and independent oracles where useful; student array restrictions do not apply to tests.


In [ ]:
def independent(output, original, before):
    np.testing.assert_array_equal(original, before)
    assert not np.shares_memory(output, original)

def check_runs(text):
    runs = encode_runs(text)
    assert isinstance(runs, list)
    assert all(isinstance(p, tuple) and len(p)==2 and isinstance(p[0],str)
               and len(p[0])==1 and type(p[1]) is int and p[1]>0 for p in runs)
    assert all(runs[i][0] != runs[i+1][0] for i in range(len(runs)-1))
    before = runs.copy()
    assert decode_runs(runs) == text
    assert runs == before

def check_frame(h, w):
    raw = np.arange(h*w, dtype=np.int64)
    before = raw.copy()
    result = sample_frame(raw,h,w)
    expected = [[int(raw[r*w+c]) for c in reversed(range(0,w,2))]
                for r in range(0,h,2)]
    np.testing.assert_array_equal(result,expected)
    assert result.shape == ((h+1)//2,(w+1)//2) and result.dtype == np.int64
    independent(result,raw,before)
    result[0,0] = -99
    np.testing.assert_array_equal(raw,before)

def bottom_up_count(n, steps):
    table = [0]*(n+1)
    table[0] = 1
    for distance in range(1,n+1):
        table[distance] = sum(table[distance-s] for s in steps if s<=distance)
    return table[n]

def check_count(n, steps):
    result = count_steps(n,steps)
    assert type(result) is int
    assert result == bottom_up_count(n,steps)

def check_episodes(temperatures, threshold, min_length):
    before_list = temperatures.copy()
    data = np.array(temperatures, dtype=np.float64)
    before_data = data.copy()
    # Independent exhaustive interval oracle: only maximal all-hot intervals.
    expected = []
    n = len(temperatures)
    for start in range(n):
        for stop in range(start + min_length, n + 1):
            if (all(v >= threshold for v in temperatures[start:stop])
                and (start == 0 or temperatures[start-1] < threshold)
                and (stop == n or temperatures[stop] < threshold)):
                expected.append((start, stop))
    py = episodes_python(temperatures, threshold, min_length)
    starts, stops = episodes_numpy(data, threshold, min_length)
    assert py == expected and temperatures == before_list
    np.testing.assert_array_equal(starts, [p[0] for p in expected])
    np.testing.assert_array_equal(stops, [p[1] for p in expected])
    for output in (starts, stops):
        assert output.dtype == np.int64 and output.ndim == 1
        independent(output, data, before_data)

def peaks_oracle(heights):
    result = np.zeros(heights.shape, dtype=bool)
    for row in range(1, heights.shape[0]-1):
        for col in range(1, heights.shape[1]-1):
            if all(heights[row,col] > heights[row+dr,col+dc]
                   for dr in (-1,0,1) for dc in (-1,0,1)
                   if (dr,dc) != (0,0)):
                result[row,col] = True
    return result

def check_peaks(heights):
    before = heights.copy()
    result = local_peaks(heights)
    np.testing.assert_array_equal(result, peaks_oracle(heights))
    assert result.dtype == np.bool_ and result.shape == heights.shape
    independent(result, heights, before)

assert encode_runs('aaabb aa') == [('a',3),('b',2),(' ',1),('a',2)]
assert decode_runs([('a',1),('a',2)]) == 'aaa'
for text in ('','x','ababab','aAAa','aaabb aa','  ','\n\nA','ééA'):
    check_runs(text)
for h,w in ((4,5),(3,4),(1,1),(1,6),(6,1)):
    check_frame(h,w)
np.testing.assert_array_equal(sample_frame(np.arange(20,dtype=np.int64),4,5),[[4,2,0],[14,12,10]])
assert count_steps(4,(1,2)) == 5
assert count_steps(0,(2,3)) == 1
assert count_steps(1,(2,3)) == 0
assert count_steps(4,(2,)) == 1
assert count_steps(100,(1,2)) == 573147844013817084101
for steps in ((1,2),(3,1,2),(2,4),(10,),(2,3)):
    for n in (0,1,4,10,100):
        check_count(n,steps)

# Instrument the explicit cache to confirm one write per nonnegative state,
# including zero results; cache hits do not recompute them.
class CountWrites(dict):
    def __init__(self):
        super().__init__({0:1})
        self.writes = {}
    def __setitem__(self, key, value):
        self.writes[key] = self.writes.get(key,0)+1
        super().__setitem__(key,value)

cache = CountWrites()
assert _count_remaining(9,(2,4),cache) == 0
assert all(v==1 for v in cache.writes.values())
assert any(v==0 for v in cache.values())
writes = cache.writes.copy()
assert _count_remaining(9,(2,4),cache)==0 and writes==cache.writes

temperatures = [70.,80.,85.,72.,90.,91.,92.,65.,88.]
assert episodes_python(temperatures,80.,2) == [(1,3),(4,7)]
for readings in (temperatures, [], [80.], [70.], [80.,80.,80.],
                 [70.,70.], [80.,70.,80.], [70.,80.,80.]):
    for minimum in (1,2,3,20):
        check_episodes(readings,80.,minimum)
terrain = np.array([[0.,0.,0.,0.,0.], [0.,9.,1.,8.,0.],
                    [0.,1.,2.,1.,0.], [0.,0.,0.,0.,0.]])
expected_peaks = np.zeros(terrain.shape, dtype=bool)
expected_peaks[1,1] = expected_peaks[1,3] = True
np.testing.assert_array_equal(local_peaks(terrain), expected_peaks)
check_peaks(terrain)
tied = terrain.copy()
tied[1,2] = 9.
np.testing.assert_array_equal(local_peaks(tied), np.zeros(tied.shape,dtype=bool))
check_peaks(tied)
for shape in ((0,0),(0,4),(4,0),(1,5),(5,2),(3,3),(4,7)):
    check_peaks(np.zeros(shape,dtype=np.float64))
check_peaks(terrain - 20)
check_peaks(terrain.T)
check_peaks(terrain[::-1, ::-1])
# Every direction must be considered; a high border neighbor blocks the center.
for dr in (-1,0,1):
    for dc in (-1,0,1):
        if (dr,dc) != (0,0):
            grid = np.zeros((3,3),dtype=np.float64)
            grid[1,1] = 5.
            grid[1+dr,1+dc] = 6.
            assert not local_peaks(grid).any()
            check_peaks(grid)

rng = np.random.default_rng(326)
for _ in range(40):
    check_runs(''.join(rng.choice(list('ab A'),size=30)))
    check_frame(int(rng.integers(1,10)),int(rng.integers(1,10)))
    steps = tuple(int(s) for s in rng.choice(np.arange(1,8),size=3,replace=False))
    check_count(int(rng.integers(0,30)),steps)
    readings = list(map(float,rng.integers(65,96,size=12)))
    check_episodes(readings,80.,int(rng.integers(1,15)))
    shape = (int(rng.integers(0,10)), int(rng.integers(0,10)))
    check_peaks(rng.integers(-10,11,size=shape).astype(np.float64))
print('PASS: five questions; examples, edge cases, cache instrumentation, and 40 seeded random cases each.')
